# Your first machine learning model

In this lesson, you will explore flower measurements and train a model to predict an Iris flower's species. Run cells in order. Pause at the **Your turn** questions to make a prediction or change the code before moving on.

We will use a held-out test set: the model learns from the training data, then we measure it on examples it did not train on. Keeping these sets separate helps us estimate how well the model may work on new data.

## 1. Import our tools

Pandas helps us inspect tabular data, Matplotlib makes plots, and scikit-learn provides the dataset, train/test split, model, and evaluation tools.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Load and inspect the data

The Iris dataset has 150 flowers, four measurements (in centimeters), and three species. It is bundled with scikit-learn; attribution and license details are in `data/README.md`.

In [ ]:
iris = load_iris(as_frame=True)
flowers = iris.frame
flowers.head()

In [ ]:
print(f"Rows: {len(flowers)}")
print(f"Measurements: {', '.join(iris.feature_names)}")
print(f"Species: {', '.join(iris.target_names)}")
flowers.describe().round(2)

**Your turn:** Which measurement has the largest average value? Do you notice anything about the ranges or standard deviations?

## 3. Visualize two measurements

Color points by species. Do the groups look easy to separate using these two measurements alone?

In [ ]:
for species_id, species_name in enumerate(iris.target_names):
    rows = flowers[flowers["target"] == species_id]
    plt.scatter(
        rows["petal length (cm)"],
        rows["petal width (cm)"],
        label=species_name,
    )

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Iris flowers by petal measurements")
plt.legend(title="Species")
plt.show()

**Your turn:** Which species appears easiest to distinguish? What might happen if we only used these two measurements instead of all four?

## 4. Make training and test sets

The target is the species label. We set aside 25% of rows for testing. `stratify=y` keeps the species proportions similar in both sets, and `random_state` makes the split repeatable.

In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print(f"Training examples: {len(X_train)}")
print(f"Test examples: {len(X_test)}")

## 5. Train a baseline model

Logistic regression is a classification algorithm despite the word “regression” in its name. The pipeline standardizes measurements before fitting the model. It learns the scaling from the training data only, avoiding information from the test set leaking into training.

In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=200),
)
model.fit(X_train, y_train)

## 6. Evaluate on examples the model has not seen

Accuracy is the fraction of test examples classified correctly. The classification report also shows precision, recall, and F1 score for each species.

In [ ]:
predictions = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, predictions):.1%}")
print()
print(classification_report(y_test, predictions, target_names=iris.target_names))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=iris.target_names,
    cmap="Blues",
)
plt.title("Predicted species vs. actual species")
plt.show()

**Your turn:** Which species did the model confuse, if any? Why is reporting test accuracy more useful than reporting accuracy on the training examples?

## Try a change

1. Change `test_size` to `0.4`. How does the score change? Why might it vary?
2. In the plot, try sepal length and sepal width instead of petal measurements. What changes?
3. Remove `StandardScaler()` from the pipeline and compare results. Why is scaling often useful?
4. Pick one flower row and use `model.predict(...)` to predict its species. What would you want to know before trusting a prediction for a real flower?